# Chapter 19 — Decision Trees

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(11)
n = 1470
tenure   = np.clip(rng.gamma(2.2, 3.0, n), 0.2, 40).round(1)
salary   = np.clip(rng.normal(65000, 18000, n), 25000, 160000).round(-2)
overtime = (rng.random(n) < 0.28).astype(int)
commute  = np.clip(rng.gamma(2.0, 6.0, n), 1, 60).round(0)
satis    = np.clip(rng.normal(3.3, 0.95, n), 1, 5).round(1)
promo    = np.clip(rng.gamma(1.6, 1.6, n), 0, 15).round(1)
dept     = rng.choice(["Sales", "R&D", "Support"], n, p=[0.32, 0.45, 0.23])
z = (-2.55 + 1.15*overtime - 0.135*tenure - 0.60*(satis - 3.3)
     + 0.024*commute + 0.095*promo - 0.000014*(salary - 65000)
     + np.where(dept == "Sales", 0.45,
                np.where(dept == "Support", 0.20, 0.0)))
left = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
hr = pd.DataFrame({"Department": dept, "YearsAtCompany": tenure,
    "MonthlyIncome": (salary/12).round(0),
    "OverTime": np.where(overtime == 1, "Yes", "No"),
    "CommuteMinutes": commute, "JobSatisfaction": satis,
    "YearsSincePromotion": promo, "Attrition": left})
hr.to_csv("hr.csv", index=False)
print(f"{len(hr):,} employees, attrition rate {hr['Attrition'].mean():.1%}")

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
hr = pd.read_csv("hr.csv")
X = pd.get_dummies(hr.drop(columns="Attrition"),
                   columns=["Department", "OverTime"],
                   drop_first=True).astype(float)
y = hr["Attrition"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      random_state=7, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
def entropy(p):
    if p in (0.0, 1.0):
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

H_parent = entropy(ytr.mean())
print(f"parent: n={len(ytr)}  p={ytr.mean():.4f}  H={H_parent:.4f}")

ot = Xtr["OverTime_Yes"].values == 1
weighted = 0.0
for label, mask in [("OverTime=Yes", ot), ("OverTime=No", ~ot)]:
    p_child = ytr[mask].mean()
    h = entropy(p_child)
    weighted += mask.sum() / len(ytr) * h
    print(f"  {label:<13} n={mask.sum():>4}  p={p_child:.4f}  H={h:.4f}")

print(f"weighted child entropy {weighted:.4f}")
print(f"information gain       {H_parent - weighted:.4f}")

### Block 2  (`c2.py`)

In [ ]:
# Is overtime really the best available split? Check every feature.
def entropy(p):
    return 0.0 if p in (0.0, 1.0) else -(p*np.log2(p) + (1-p)*np.log2(1-p))

def gain(col, thresh):
    left = Xtr[col].values <= thresh
    if left.sum() == 0 or (~left).sum() == 0:
        return 0.0
    w = sum(m.sum()/len(ytr) * entropy(ytr[m].mean()) for m in (left, ~left))
    return entropy(ytr.mean()) - w

print(f"{'feature':<22}{'best threshold':>16}{'gain':>9}")
best = []
for col in Xtr.columns:
    vals = np.unique(Xtr[col].values)
    cands = (vals[:-1] + vals[1:]) / 2 if len(vals) > 2 else [vals.mean()]
    g, t = max((gain(col, t), t) for t in cands)
    best.append((g, col, t))
    print(f"{col:<22}{t:>16.2f}{g:>9.4f}")
g, col, t = max(best)
print(f"\nwinner: {col} at {t:.2f}, gain {g:.4f}")

### Block 3  (`c3.py`)

In [ ]:
# The criterion must match what you computed. sklearn defaults to gini.
for crit in ("entropy", "gini"):
    t = DecisionTreeClassifier(max_depth=1, criterion=crit,
                               random_state=0).fit(Xtr, ytr)
    f = Xtr.columns[t.tree_.feature[0]]
    print(f"criterion={crit:<8} root: {f} <= {t.tree_.threshold[0]:.2f}")

print()
# Gini and entropy rank splits almost identically, which is why the default
# rarely matters -- but comparing one against the other does.
def gini(p):
    return 1 - (p**2 + (1-p)**2)
def entropy(p):
    return 0.0 if p in (0.0, 1.0) else -(p*np.log2(p) + (1-p)*np.log2(1-p))
print(f"{'p':>6}{'entropy':>10}{'gini':>8}")
for p in (0.0, 0.1215, 0.25, 0.5, 0.75):
    print(f"{p:>6.4f}{entropy(p):>10.4f}{gini(p):>8.4f}")

### Block 4  (`c4.py`)

In [ ]:
# A tree left alone memorizes. Pruning is not optional.
print(f"{'setting':<26}{'leaves':>8}{'train':>9}{'CV AUC':>9}{'sd':>8}")
settings = [("unlimited", {}),
            ("max_depth=3", {"max_depth": 3}),
            ("max_depth=4", {"max_depth": 4}),
            ("min_samples_leaf=20", {"min_samples_leaf": 20}),
            ("min_samples_leaf=50", {"min_samples_leaf": 50}),
            ("ccp_alpha=0.002", {"ccp_alpha": 0.002})]
for name, kw in settings:
    m = DecisionTreeClassifier(random_state=0, **kw).fit(Xtr, ytr)
    s = cross_val_score(DecisionTreeClassifier(random_state=0, **kw),
                        Xtr, ytr, cv=cv, scoring="roc_auc")
    tr = roc_auc_score(ytr, m.predict_proba(Xtr)[:, 1])
    print(f"{name:<26}{m.get_n_leaves():>8}{tr:>9.4f}"
          f"{s.mean():>9.4f}{s.std():>8.4f}")

### Block 5  (`c5.py`)

In [ ]:
t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(export_text(t, feature_names=list(Xtr.columns), decimals=1))

### Block 6  (`c6.py`)

In [ ]:
t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
leaf = t.apply(Xtr)                       # which leaf each employee lands in

rows = []
for lf in np.unique(leaf):
    m = leaf == lf
    rows.append((m.sum(), ytr[m].mean(), lf))
rows.sort(key=lambda r: -r[1])

print(f"{'leaf':>6}{'employees':>11}{'attrition':>11}{'vs base':>9}")
base = ytr.mean()
for n, p, lf in rows:
    print(f"{lf:>6}{n:>11}{p:>11.1%}{p/base:>8.1f}x")

top = rows[0]
print(f"\nhighest-risk leaf holds {top[0]} of {len(ytr)} employees "
      f"({top[0]/len(ytr):.1%})")
print(f"and {top[1]:.0%} of them left, against a {base:.1%} base rate")